In [1]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pymongo import MongoClient, UpdateOne
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
client = MongoClient(os.getenv("MONGO_CONNECTION_STRING"))
db = client["final_project"]

# Read savant
raw = db["savant"]
df = pd.DataFrame(list(raw.find({})))
df = df.replace("", pd.NA)

print(f"Loaded {len(df)} pitch records")

# 1. Build games collection
game_cols = ["game_pk", "game_date", "game_type", "game_year", "home_team", "away_team"]
games_df = df[game_cols].drop_duplicates(subset=["game_pk"])

games_col = db["games"]
games_col.drop()  # fresh start
game_ops = [
    UpdateOne({"game_pk": row["game_pk"]}, {"$setOnInsert": row.to_dict()}, upsert=True)
    for _, row in games_df.iterrows()
]
result = games_col.bulk_write(game_ops)
print(f"Games inserted: {result.upserted_count}")

# 2. Build players collection
# Pitchers
pitcher_cols = [
    "pitcher", "player_name", "p_throws",
    "age_pit", "age_pit_legacy",
    "pitcher_days_since_prev_game", "pitcher_days_until_next_game"
]
pitchers_df = df[pitcher_cols].drop_duplicates(subset=["pitcher"]).rename(
    columns={
        "pitcher": "player_id",
        "p_throws": "throws",
        "age_pit": "age",
        "age_pit_legacy": "age_legacy",
        "pitcher_days_since_prev_game": "days_since_prev_game",
        "pitcher_days_until_next_game": "days_until_next_game"
    }
)
pitchers_df["role"] = "pitcher"

# Batters player_name is pitcher name in Statcast, so batters won't have names here
# We store what we have a real production project might join another source for batter names
batter_cols = [
    "batter", "stand",
    "age_bat", "age_bat_legacy",
    "batter_days_since_prev_game", "batter_days_until_next_game"
]
batters_df = df[batter_cols].drop_duplicates(subset=["batter"]).rename(
    columns={
        "batter": "player_id",
        "stand": "bats",
        "age_bat": "age",
        "age_bat_legacy": "age_legacy",
        "batter_days_since_prev_game": "days_since_prev_game",
        "batter_days_until_next_game": "days_until_next_game"
    }
)
batters_df["role"] = "batter"

# Combine a player_id that appears as both pitcher and batter gets two docs
players_col = db["players"]
players_col.drop()

all_players = pd.concat([pitchers_df, batters_df], ignore_index=True)
player_ops = [
    UpdateOne(
        {"player_id": row["player_id"], "role": row["role"]},
        {"$setOnInsert": {k: v for k, v in row.items() if pd.notna(v)}},
        upsert=True
    )
    for _, row in all_players.iterrows()
]
result = players_col.bulk_write(player_ops)
print(f"Players inserted: {result.upserted_count}")

# 3. Build pitches collection
# Drop fields that now live in other collections FKs
drop_from_pitches = [
    "game_date", "game_type", "game_year", "home_team", "away_team",  # games
    "player_name", "p_throws", "stand",                               # players
    "age_pit", "age_pit_legacy", "age_bat", "age_bat_legacy",
    "pitcher_days_since_prev_game", "pitcher_days_until_next_game",
    "batter_days_since_prev_game", "batter_days_until_next_game",
]
pitches_df = df.drop(columns=[c for c in drop_from_pitches if c in df.columns])

# Convert _id ObjectId to string so we don't duplicate
pitches_df["_id"] = pitches_df["_id"].astype(str)

pitches_col = db["pitches"]
pitches_col.drop()
pitches_col.insert_many(pitches_df.to_dict("records"))
print(f"Pitches inserted: {len(pitches_df)}")

# 4. Add indexes for fast $lookup / query
pitches_col.create_index("game_pk")
pitches_col.create_index("pitcher")
pitches_col.create_index("batter")
pitches_col.create_index([("pitcher", 1), ("pitch_type", 1)])
pitches_col.create_index([("events", 1)])
games_col.create_index("game_pk", unique=True)
players_col.create_index([("player_id", 1), ("role", 1)], unique=True)


Loaded 25000 pitch records
Games inserted: 92
Players inserted: 762
Pitches inserted: 25000


'player_id_1_role_1'

# 6 Queries based on the data

## 1. Which pitchers appeared the most in the pitches dataset last season?

In [3]:
pipeline = [
    {
        "$lookup": {
            "from": "players",
            "let": {"pid": "$pitcher"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [
                                {"$eq": ["$player_id", "$$pid"]},
                                {"$eq": ["$role", "pitcher"]}
                            ]
                        }
                    }
                }
            ],
            "as": "pitcher_info"
        }
    },
    {"$unwind": "$pitcher_info"},

    {
        "$group": {
            "_id": "$pitcher_info.player_name",
            "total_pitches": {"$sum": 1}
        }
    },

    {"$sort": {"total_pitches": -1}},
    {"$limit": 25}
]

list(db.pitches.aggregate(pipeline))

[{'_id': 'Yesavage, Trey', 'total_pitches': 538},
 {'_id': 'Snell, Blake', 'total_pitches': 537},
 {'_id': 'Gausman, Kevin', 'total_pitches': 529},
 {'_id': 'Yamamoto, Yoshinobu', 'total_pitches': 526},
 {'_id': 'Bieber, Shane', 'total_pitches': 378},
 {'_id': 'Glasnow, Tyler', 'total_pitches': 378},
 {'_id': 'Kirby, George', 'total_pitches': 363},
 {'_id': 'Ohtani, Shohei', 'total_pitches': 336},
 {'_id': 'Gilbert, Logan', 'total_pitches': 331},
 {'_id': 'Peralta, Freddy', 'total_pitches': 320},
 {'_id': 'Skubal, Tarik', 'total_pitches': 303},
 {'_id': 'Schlittler, Cam', 'total_pitches': 292},
 {'_id': 'Varland, Louis', 'total_pitches': 269},
 {'_id': 'Miller, Bryce', 'total_pitches': 261},
 {'_id': 'Sánchez, Cristopher', 'total_pitches': 249},
 {'_id': 'Misiorowski, Jacob', 'total_pitches': 242},
 {'_id': 'Domínguez, Seranthony', 'total_pitches': 223},
 {'_id': 'Scherzer, Max', 'total_pitches': 220},
 {'_id': 'Taillon, Jameson', 'total_pitches': 218},
 {'_id': 'Hoffman, Jeff', 'total

# 2. Hits allowed on Blake Snell fastballs

In [4]:
pipeline = [
    {
        "$lookup": {
            "from": "players",
            "let": {"pid": "$pitcher"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$eq": ["$player_id", "$$pid"]
                        }
                    }
                }
            ],
            "as": "pitcher_info"
        }
    },

    {"$unwind": "$pitcher_info"},

    {
        "$match": {
            "pitcher_info.player_name": "Snell, Blake",
            "pitch_type": {"$in": ["FF", "SI"]},
            "events": {"$in": ["single", "double", "triple", "home_run"]}
        }
    },

    {
        "$group": {
            "_id": {
                "pitch": "$pitch_name",
                "result": "$events"
            },
            "count": {"$sum": 1},
            "avg_launch_speed": {"$avg": "$launch_speed"},
            "avg_launch_angle": {"$avg": "$launch_angle"}
        }
    },

    {"$sort": {"count": -1}}
]

list(db.pitches.aggregate(pipeline))

[{'_id': {'pitch': '4-Seam Fastball', 'result': 'home_run'},
  'count': 3,
  'avg_launch_speed': 103.63333333333333,
  'avg_launch_angle': 27.666666666666668},
 {'_id': {'pitch': '4-Seam Fastball', 'result': 'single'},
  'count': 3,
  'avg_launch_speed': 82.53333333333333,
  'avg_launch_angle': -2.0},
 {'_id': {'pitch': '4-Seam Fastball', 'result': 'double'},
  'count': 2,
  'avg_launch_speed': 102.75,
  'avg_launch_angle': 8.5}]

# 3. Strikeout rate by count (balls-strikes) across all pitchers

In [5]:
pipeline = [
    {
        "$match": {
            "events": {"$exists": True, "$ne": ""}
        }
    },

    {
        "$group": {
            "_id": {
                "balls": "$balls",
                "strikes": "$strikes"
            },
            "total_pas": {"$sum": 1},
            "strikeouts": {
                "$sum": {
                    "$cond": [{"$eq": ["$events", "strikeout"]}, 1, 0]
                }
            },
            "walks": {
                "$sum": {
                    "$cond": [{"$eq": ["$events", "walk"]}, 1, 0]
                }
            },
            "hits": {
                "$sum": {
                    "$cond": [
                        {"$in": ["$events", ["single","double","triple","home_run"]]},
                        1,
                        0
                    ]
                }
            }
        }
    },

    {
        "$addFields": {
            "k_rate": {"$round": [{"$divide": ["$strikeouts", "$total_pas"]}, 3]},
            "bb_rate": {"$round": [{"$divide": ["$walks", "$total_pas"]}, 3]}
        }
    },

    {"$sort": {"_id.balls": 1, "_id.strikes": 1}}
]

list(db.pitches.aggregate(pipeline))

[{'_id': {'balls': 0, 'strikes': 0},
  'total_pas': 6312,
  'strikeouts': 0,
  'walks': 0,
  'hits': 216,
  'k_rate': 0.0,
  'bb_rate': 0.0},
 {'_id': {'balls': 0, 'strikes': 1},
  'total_pas': 3251,
  'strikeouts': 0,
  'walks': 0,
  'hits': 165,
  'k_rate': 0.0,
  'bb_rate': 0.0},
 {'_id': {'balls': 0, 'strikes': 2},
  'total_pas': 1707,
  'strikeouts': 325,
  'walks': 0,
  'hits': 96,
  'k_rate': 0.19,
  'bb_rate': 0.0},
 {'_id': {'balls': 1, 'strikes': 0},
  'total_pas': 2378,
  'strikeouts': 0,
  'walks': 0,
  'hits': 117,
  'k_rate': 0.0,
  'bb_rate': 0.0},
 {'_id': {'balls': 1, 'strikes': 1},
  'total_pas': 2567,
  'strikeouts': 0,
  'walks': 0,
  'hits': 167,
  'k_rate': 0.0,
  'bb_rate': 0.0},
 {'_id': {'balls': 1, 'strikes': 2},
  'total_pas': 2463,
  'strikeouts': 497,
  'walks': 0,
  'hits': 121,
  'k_rate': 0.202,
  'bb_rate': 0.0},
 {'_id': {'balls': 2, 'strikes': 0},
  'total_pas': 799,
  'strikeouts': 0,
  'walks': 0,
  'hits': 23,
  'k_rate': 0.0,
  'bb_rate': 0.0},
 {

# 4.  Home run rate by pitch type (with game context via $lookup)

In [6]:
pipeline = [
    {
        "$match": {
            "events": {"$exists": True, "$ne": ""}
        }
    },

    {
        "$lookup": {
            "from": "games",
            "localField": "game_pk",
            "foreignField": "game_pk",
            "as": "game_info"
        }
    },

    {"$unwind": "$game_info"},

    {
        "$group": {
            "_id": {
                "pitch_type": "$pitch_type",
                "pitch_name": "$pitch_name"
            },
            "total": {"$sum": 1},
            "home_runs": {
                "$sum": {
                    "$cond": [{"$eq": ["$events", "home_run"]}, 1, 0]
                }
            }
        }
    },

    {
        "$addFields": {
            "hr_rate": {"$round": [{"$divide": ["$home_runs", "$total"]}, 4]}
        }
    },

    {"$match": {"total": {"$gte": 20}}},
    {"$sort": {"hr_rate": -1}},
    {"$limit": 10}
]

list(db.pitches.aggregate(pipeline))

[{'_id': {'pitch_type': 'ST', 'pitch_name': 'Sweeper'},
  'total': 1534,
  'home_runs': 18,
  'hr_rate': 0.0117},
 {'_id': {'pitch_type': 'FF', 'pitch_name': '4-Seam Fastball'},
  'total': 8277,
  'home_runs': 85,
  'hr_rate': 0.0103},
 {'_id': {'pitch_type': 'SL', 'pitch_name': 'Slider'},
  'total': 3879,
  'home_runs': 34,
  'hr_rate': 0.0088},
 {'_id': {'pitch_type': 'CU', 'pitch_name': 'Curveball'},
  'total': 1781,
  'home_runs': 15,
  'hr_rate': 0.0084},
 {'_id': {'pitch_type': 'CH', 'pitch_name': 'Changeup'},
  'total': 2302,
  'home_runs': 18,
  'hr_rate': 0.0078},
 {'_id': {'pitch_type': 'SI', 'pitch_name': 'Sinker'},
  'total': 3538,
  'home_runs': 27,
  'hr_rate': 0.0076},
 {'_id': {'pitch_type': 'SV', 'pitch_name': 'Slurve'},
  'total': 132,
  'home_runs': 1,
  'hr_rate': 0.0076},
 {'_id': {'pitch_type': 'FC', 'pitch_name': 'Cutter'},
  'total': 1460,
  'home_runs': 9,
  'hr_rate': 0.0062},
 {'_id': {'pitch_type': 'FS', 'pitch_name': 'Split-Finger'},
  'total': 1343,
  'hom

# 5. Pitcher arsenal effectiveness: whiff rate per pitch type

In [7]:
pipeline = [
    {
        "$lookup": {
            "from": "players",
            "let": {"pid": "$pitcher"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {"$eq": ["$player_id", "$$pid"]}
                    }
                }
            ],
            "as": "pitcher_info"
        }
    },

    {"$unwind": "$pitcher_info"},

    {
        "$group": {
            "_id": {
                "pitcher_name": "$pitcher_info.player_name",
                "pitch_name": "$pitch_name"
            },
            "pitches_thrown": {"$sum": 1},
            "swinging_strikes": {
                "$sum": {
                    "$cond": [
                        {"$in": ["$description", ["swinging_strike", "swinging_strike_blocked"]]},
                        1,
                        0
                    ]
                }
            },
            "avg_velocity": {"$avg": "$release_speed"}
        }
    },

    {
        "$addFields": {
            "whiff_rate": {
                "$round": [
                    {"$divide": ["$swinging_strikes", "$pitches_thrown"]},
                    3
                ]
            }
        }
    },

    {"$match": {"pitches_thrown": {"$gte": 15}}},
    {"$sort": {"whiff_rate": -1}},
    {"$limit": 15}
]

list(db.pitches.aggregate(pipeline))

[{'_id': {'pitcher_name': 'De León, José', 'pitch_name': 'Slurve'},
  'pitches_thrown': 24,
  'swinging_strikes': 11,
  'avg_velocity': 74.59166666666667,
  'whiff_rate': 0.458},
 {'_id': {'pitcher_name': 'Bradish, Kyle', 'pitch_name': 'Slider'},
  'pitches_thrown': 30,
  'swinging_strikes': 12,
  'avg_velocity': 87.44666666666667,
  'whiff_rate': 0.4},
 {'_id': {'pitcher_name': 'Skubal, Tarik', 'pitch_name': 'Changeup'},
  'pitches_thrown': 77,
  'swinging_strikes': 30,
  'avg_velocity': 88.88441558441559,
  'whiff_rate': 0.39},
 {'_id': {'pitcher_name': 'Latz, Jacob', 'pitch_name': 'Changeup'},
  'pitches_thrown': 18,
  'swinging_strikes': 7,
  'avg_velocity': 84.41666666666667,
  'whiff_rate': 0.389},
 {'_id': {'pitcher_name': 'Woods Richardson, Simeon',
   'pitch_name': 'Split-Finger'},
  'pitches_thrown': 21,
  'swinging_strikes': 8,
  'avg_velocity': 85.57619047619048,
  'whiff_rate': 0.381},
 {'_id': {'pitcher_name': 'Ragans, Cole', 'pitch_name': 'Changeup'},
  'pitches_thrown':

# 6. Batter performance against left vs. right handed pitchers

In [8]:
pipeline = [
    {
        "$match": {
            "events": {"$exists": True, "$ne": ""}
        }
    },

    {
        "$lookup": {
            "from": "players",
            "let": {"bid": "$batter"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {"$eq": ["$player_id", "$$bid"]}
                    }
                }
            ],
            "as": "batter_info"
        }
    },

    {"$unwind": "$batter_info"},

    {
        "$lookup": {
            "from": "players",
            "let": {"pid": "$pitcher"},
            "pipeline": [
                {
                    "$match": {
                        "$expr": {"$eq": ["$player_id", "$$pid"]}
                    }
                }
            ],
            "as": "pitcher_info"
        }
    },

    {"$unwind": "$pitcher_info"},

    {
        "$group": {
            "_id": {
                "batter_side": "$batter_info.bats",
                "pitcher_hand": "$pitcher_info.throws"
            },
            "plate_appearances": {"$sum": 1},
            "hits": {
                "$sum": {
                    "$cond": [
                        {"$in": ["$events", ["single","double","triple","home_run"]]},
                        1,
                        0
                    ]
                }
            }
        }
    },

    {
        "$addFields": {
            "batting_avg": {
                "$round": [
                    {"$divide": ["$hits", "$plate_appearances"]},
                    3
                ]
            }
        }
    }
]

list(db.pitches.aggregate(pipeline))

[{'_id': {'batter_side': 'R'},
  'plate_appearances': 162,
  'hits': 9,
  'batting_avg': 0.056},
 {'_id': {'batter_side': 'R', 'pitcher_hand': 'L'},
  'plate_appearances': 3987,
  'hits': 241,
  'batting_avg': 0.06},
 {'_id': {'pitcher_hand': 'L'},
  'plate_appearances': 135,
  'hits': 7,
  'batting_avg': 0.052},
 {'_id': {'batter_side': 'L', 'pitcher_hand': 'R'},
  'plate_appearances': 8712,
  'hits': 422,
  'batting_avg': 0.048},
 {'_id': {'batter_side': 'L'},
  'plate_appearances': 174,
  'hits': 7,
  'batting_avg': 0.04},
 {'_id': {'batter_side': 'L', 'pitcher_hand': 'L'},
  'plate_appearances': 3049,
  'hits': 145,
  'batting_avg': 0.048},
 {'_id': {'batter_side': 'R', 'pitcher_hand': 'R'},
  'plate_appearances': 9252,
  'hits': 485,
  'batting_avg': 0.052},
 {'_id': {'pitcher_hand': 'R'},
  'plate_appearances': 237,
  'hits': 14,
  'batting_avg': 0.059}]

In [10]:
from pymongo import MongoClient
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()
client = MongoClient(os.getenv("MONGO_CONNECTION_STRING"))
db = client["final_project"]

# Use the flat collection
raw = db["savant"]
df = pd.DataFrame(list(raw.find({})))
df.head()

,_id,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
0,69d3c54eef428760b51381cf,SL,11/1/2025,85.9,-3.1,5.64,"Scherzer, Max",500743,453286,single,...,,3.13,-0.2,-0.2,31.5,-5.823739,-17.027322,29.582052,32.972714,44.206122
1,69d3c54eef428760b51381d0,SL,11/1/2025,85.4,-3.11,5.55,"Scherzer, Max",500743,453286,,...,,2.86,-0.29,-0.29,33.7,,,,,
2,69d3c54eef428760b51381d1,CH,11/1/2025,84.8,-3.3,5.36,"Scherzer, Max",571771,453286,,...,,2.82,0.79,0.79,27.4,,,,,
3,69d3c54eef428760b51381d2,CU,11/1/2025,73.2,-3.02,5.75,"Scherzer, Max",571771,453286,strikeout,...,,5.25,-1.04,-1.04,40.4,2.410072,30.857094,31.03147,44.98723,21.325831
4,69d3c54eef428760b51381d3,SL,11/1/2025,88.7,-1.92,5.99,"Ohtani, Shohei",666182,660271,home_run,...,,2.64,-0.1,-0.1,40.8,3.753491,-9.95437,31.73651,37.886817,30.395348


In [11]:
df = df.replace("", pd.NA)

In [12]:

for column in df.columns:
    null_percentage = df[column].isna().sum() / len(df) * 100
    
    print(f"Column: {column}, Null Percentage: {null_percentage:.2f}%")

Column: _id, Null Percentage: 0.00%
Column: pitch_type, Null Percentage: 0.67%
Column: game_date, Null Percentage: 0.00%
Column: release_speed, Null Percentage: 0.67%
Column: release_pos_x, Null Percentage: 0.67%
Column: release_pos_z, Null Percentage: 0.67%
Column: player_name, Null Percentage: 0.00%
Column: batter, Null Percentage: 0.00%
Column: pitcher, Null Percentage: 0.00%
Column: events, Null Percentage: 74.59%
Column: description, Null Percentage: 0.00%
Column: spin_dir, Null Percentage: 100.00%
Column: spin_rate_deprecated, Null Percentage: 100.00%
Column: break_angle_deprecated, Null Percentage: 100.00%
Column: break_length_deprecated, Null Percentage: 100.00%
Column: zone, Null Percentage: 0.67%
Column: des, Null Percentage: 0.00%
Column: game_type, Null Percentage: 0.00%
Column: stand, Null Percentage: 0.00%
Column: p_throws, Null Percentage: 0.00%
Column: home_team, Null Percentage: 0.00%
Column: away_team, Null Percentage: 0.00%
Column: type, Null Percentage: 0.00%
Column

In [13]:
# converting on_1b - on_3b to boolean
df['on_1b'] = df['on_1b'].notna().astype(int)
df['on_2b'] = df['on_2b'].notna().astype(int)
df['on_3b'] = df['on_3b'].notna().astype(int)

In [14]:
# Important features
features = [
    # Player context
    "pitcher",
    "p_throws",
    "stand",

    # Count
    "balls",
    "strikes",

    # Game state
    "outs_when_up",
    "inning",
    "inning_topbot",

    # Base runners (binary)
    "on_1b",
    "on_2b",
    "on_3b",

    # Optional context
    "home_score_diff",
    "n_thruorder_pitcher",

    # Potential features
    "release_speed",
    "release_spin_rate",
    "spin_axis"
]

In [15]:
ml_df = df[features].copy()

In [16]:
# ml_df.head()

In [17]:
# ml_df['p_throws'].value_counts()
# ml_df['stand'].value_counts()
# ml_df['inning_topbot'].value_counts()

In [18]:
# One hot encoding categorical features
ml_df = pd.get_dummies(ml_df, columns=['p_throws', 'stand', 'inning_topbot'], drop_first=True)
ml_df['p_throws_R'] = ml_df['p_throws_R'].astype(int)
ml_df['stand_R'] = ml_df['stand_R'].astype(int)
ml_df['inning_topbot_Top'] = ml_df['inning_topbot_Top'].astype(int)

In [19]:
for col in ml_df.columns:
    print(f"{col}: {sum(ml_df[col].isna()) / len(ml_df) * 100:.2f}% null values")

pitcher: 0.00% null values
balls: 0.00% null values
strikes: 0.00% null values
outs_when_up: 0.00% null values
inning: 0.00% null values
on_1b: 0.00% null values
on_2b: 0.00% null values
on_3b: 0.00% null values
home_score_diff: 0.00% null values
n_thruorder_pitcher: 0.00% null values
release_speed: 0.67% null values
release_spin_rate: 0.70% null values
spin_axis: 0.70% null values
p_throws_R: 0.00% null values
stand_R: 0.00% null values
inning_topbot_Top: 0.00% null values


In [20]:
# filling null values with median for numeric features based on pitcher
cols_to_fill = ['release_speed', 'release_spin_rate', 'spin_axis']
for col in cols_to_fill:
    ml_df[col] = ml_df.groupby('pitcher')[col].transform(lambda x: x.fillna(x.median()))

C:\Users\linkm\AppData\Local\Temp\ipykernel_19956\1014031560.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ml_df[col] = ml_df.groupby('pitcher')[col].transform(lambda x: x.fillna(x.median()))
C:\Users\linkm\AppData\Local\Temp\ipykernel_19956\1014031560.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ml_df[col] = ml_df.groupby('pitcher')[col].transform(lambda x: x.fillna(x.median()))
C:\Users\linkm\AppData\Local\Temp\ipykernel_19956\1014031560.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will 

In [21]:
ml_df.columns

Index(['pitcher', 'balls', 'strikes', 'outs_when_up', 'inning', 'on_1b',
       'on_2b', 'on_3b', 'home_score_diff', 'n_thruorder_pitcher',
       'release_speed', 'release_spin_rate', 'spin_axis', 'p_throws_R',
       'stand_R', 'inning_topbot_Top'],
      dtype='object')

In [27]:
# Once model is trained, we can save it using joblib
import joblib
joblib.dump(model, 'pitch_prediction_model.pkl')

['pitch_prediction_model.pkl']

In [26]:
import joblib

numeric_expected = [
    "release_speed", "release_spin_rate", "spin_axis",
    "pfx_x", "pfx_z", "vx0", "vy0", "vz0", "ax", "ay", "az",
    "release_pos_x", "release_pos_y", "release_pos_z", "release_extension",
    "balls", "strikes", "outs_when_up", "inning",
    "on_1b", "on_2b", "on_3b", "home_score_diff", "n_thruorder_pitcher",
]
joblib.dump(numeric_expected, "backend/numeric_expected.pkl")
print("done")

done
